[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Deepak9242/triton-hardware-kernels/blob/main/02_fused_swiglu/Fused_RMSNorm_%2B_SwiGLU_.ipynb)

In [ ]:
!pip install -qU triton torch

In [4]:
%%writefile profile_swiglu.py
import torch
import triton
import triton.language as tl


@triton.jit
def fused_swiglu_kernel(gate_ptr, up_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    gate = tl.load(gate_ptr + offsets, mask=mask).to(tl.float32)
    up = tl.load(up_ptr + offsets, mask=mask).to(tl.float32)

    sigmoid_gate = 1.0 / (1.0 + tl.exp(-gate))
    silu_gate = gate * sigmoid_gate
    result = silu_gate * up

    tl.store(out_ptr + offsets, result.to(tl.float16), mask=mask)

def triton_swiglu(gate, up):
    out = torch.empty_like(gate)
    n_elements = gate.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    fused_swiglu_kernel[grid](gate, up, out, n_elements, BLOCK_SIZE=1024)
    return out

# --- 2. Profile Execution ---
def main():
    gate = torch.randn(8, 2048, 4096, device='cuda', dtype=torch.float16)
    up = torch.randn(8, 2048, 4096, device='cuda', dtype=torch.float16)

    # Warmup
    _ = torch.nn.functional.silu(gate) * up
    _ = triton_swiglu(gate, up)
    torch.cuda.synchronize()

    # PyTorch Native execution
    out_torch = torch.nn.functional.silu(gate) * up
    torch.cuda.synchronize()

    # Triton execution
    out_triton = triton_swiglu(gate, up)
    torch.cuda.synchronize()

if __name__ == "__main__":
    main()

Writing profile_swiglu.py


In [ ]:
!apt-get update -y
!apt-get install -y nsight-compute nsight-systems

In [8]:
!ncu python profile_swiglu.py

==PROF== Connected to process 6709 (/usr/bin/python3.13)
==PROF== Profiling "distribution_elementwise_grid..." - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "distribution_elementwise_grid..." - 1: 0%....50%....100% - 9 passes
==PROF== Profiling "vectorized_elementwise_kernel" - 2: 0%....50%....100% - 9 passes
==PROF== Profiling "vectorized_elementwise_kernel" - 3: 0%....50%....100% - 9 passes
==PROF== Profiling "fused_swiglu_kernel" - 4: 0%....50%....100% - 9 passes
==PROF== Profiling "vectorized_elementwise_kernel" - 5: 0%....50%....100% - 9 passes
==PROF== Profiling "vectorized_elementwise_kernel" - 6: 0%....50%....100% - 9 passes
==PROF== Profiling "fused_swiglu_kernel" - 7: 0%....50%....100% - 9 passes
==PROF== Disconnected from process 6709
[6709] python3.13@127.0.0.1
  void at::<unnamed>::distribution_elementwise_grid_stride_kernel<float, 4, void templates::normal_and_transform<c10::Half, float, at::CUDAGeneratorImpl *, void templates::normal_kernel<at::CUDAGeneratorImpl *